# 第 5 章 · Tool Use（Vertex AI Search）

书里用的 `agents.VSearchAgent` **在 ADK 2.13 里已经不存在了**——现在它是**工具**：
`VertexAiSearchTool(data_store_id=...)` 挂到普通 `LlmAgent` 的 `tools=[]` 上。

正好印证本章主线：特殊能力最终都会被统一成「又一个工具」。

⚠️ 本机跑不了：需要 Vertex AI 项目 + 真实 datastore（不是 API key 能解决的）。
所以这格只在缺配置时给出说明，不产生假输出。

kernel `Python (agentic-adk)`。


In [1]:
import sys
import uuid
from pathlib import Path

# 【适配】定位仓库根；后端由 .env 的 ADK_BACKEND 决定
_root = Path.cwd().resolve()
for _d in (_root, *_root.parents):
    if (_d / "common").is_dir():
        sys.path.insert(0, str(_d))
        break
import os

from common import adk_model, short_error, retry_call
from google.adk.agents import LlmAgent
from google.adk.runners import InMemoryRunner
from google.adk.tools.vertex_ai_search_tool import VertexAiSearchTool
from google.genai import types

# 形如 projects/{project}/locations/{location}/collections/{collection}/dataStores/{id}
DATASTORE_ID = os.environ.get("DATASTORE_ID")
USER_ID = "user_123"

# 【适配】书里的 agents.VSearchAgent 已被移除，改为「普通 LlmAgent + VertexAiSearchTool」
vsearch_agent = LlmAgent(
    name="q2_strategy_vsearch_agent",
    description="Answers questions about Q2 strategy documents using Vertex AI Search.",
    model=adk_model(),                          # 【适配】gemini-2.0-flash-exp 已下线
    tools=[VertexAiSearchTool(data_store_id=DATASTORE_ID)] if DATASTORE_ID else [],
    instruction="Answer using the Vertex AI Search results. Cite where you can.",
)


def call_vsearch_agent(query: str) -> None:
    runner = InMemoryRunner(vsearch_agent)
    session_id = str(uuid.uuid4())
    # 【适配】create_session 是协程 → _sync 版
    runner.session_service.create_session_sync(
        app_name=runner.app_name, user_id=USER_ID, session_id=session_id
    )
    print(f"\nUser: {query}")
    try:
        for event in retry_call(lambda: list(runner.run(
                user_id=USER_ID, session_id=session_id,
                new_message=types.Content(role="user", parts=[types.Part(text=query)]))), label="检索 "):
            if not event.is_final_response() or not event.content:
                continue
            text = getattr(event.content, "text", None) or "".join(
                p.text for p in (event.content.parts or []) if getattr(p, "text", None))
            print("Agent:", (text or "").strip())
            gm = getattr(event, "grounding_metadata", None)
            print("  (引用来源:", len(getattr(gm, "grounding_attributions", None) or []), "个)" if gm else "(无 grounding)")
    except Exception as e:
        print(f"  ❌ {type(e).__name__}: {short_error(e)}")


if not DATASTORE_ID:
    print("⚠️ 没有 DATASTORE_ID：agent 未挂搜索工具，只能凭模型自身知识回答。")
    print("   要真跑需 Vertex AI 项目 + datastore：export DATASTORE_ID=projects/.../dataStores/...")
    print("   这就是书里这一节在本机无法复现的原因——它依赖的不是 API key，而是一整套云端资源。")
else:
    call_vsearch_agent("Summarize the main points about the Q2 strategy document.")


⚠️ 没有 DATASTORE_ID：agent 未挂搜索工具，只能凭模型自身知识回答。
   要真跑需 Vertex AI 项目 + datastore：export DATASTORE_ID=projects/.../dataStores/...
   这就是书里这一节在本机无法复现的原因——它依赖的不是 API key，而是一整套云端资源。
